# Módulo 2 — Manipulação de Dados com Pandas
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR  
**Dataset:** Brazilian Cities — Dados socioeconômicos de 5.578 municípios brasileiros

---

## 📚 O que você vai aprender neste módulo

| Seção | Conteúdo |
|-------|----------|
| 2.1 | Introdução ao Pandas — Series e DataFrames |
| 2.2 | Leitura e inspeção do dataset |
| 2.3 | Limpeza e tratamento de dados |
| 2.4 | Manipulação básica e estatística descritiva |
| 2.5 | Exercícios práticos |


> 💡 **Dica:** Execute cada célula em sequência com `Shift + Enter`


---
## 2.1 Teoria — Introdução ao Pandas

**Pandas** é a principal biblioteca Python para análise de dados. Ela fornece duas estruturas fundamentais:

- **`Series`** → vetor unidimensional (como uma coluna de planilha)
- **`DataFrame`** → tabela bidimensional (como uma planilha completa)

```
DataFrame
┌──────────────┬─────────┬─────────┐
│     CITY     │  STATE  │  IDHM   │  ← colunas (Series)
├──────────────┼─────────┼─────────┤
│  São Paulo   │   SP    │  0.805  │  ← linha (índice 0)
│  Fortaleza   │   CE    │  0.754  │  ← linha (índice 1)
│  ...         │   ...   │  ...    │
└──────────────┴─────────┴─────────┘
```


In [ ]:
# ── Importações necessárias para todo o módulo ──────────────────
import pandas as pd      # Manipulação de dados
import numpy as np       # Operações numéricas
import warnings
warnings.filterwarnings('ignore')

print(f"✅ Pandas versão: {pd.__version__}")
print(f"✅ NumPy  versão: {np.__version__}")


### Prática — Criando Series e DataFrames do zero

In [ ]:
# ── Criando uma Series ──────────────────────────────────────────
# Uma Series é basicamente uma lista com índice (rótulo)
populacao = pd.Series(
    [12_325_232, 2_703_391, 1_440_000, 3_849_322],
    index=['São Paulo', 'Fortaleza', 'Manaus', 'Salvador'],
    name='Populacao'
)

print("Series — Populações:")
print(populacao)
print(f"\nTipo: {type(populacao)}")
print(f"Dtype: {populacao.dtype}")


In [ ]:
# ── Acessando elementos de uma Series ───────────────────────────
print("Fortaleza:", populacao['Fortaleza'])         # pelo rótulo
print("Primeiro elemento:", populacao.iloc[0])       # pela posição
print("\nCidades com pop > 2 milhões:")
print(populacao[populacao > 2_000_000])


In [ ]:
# ── Criando um DataFrame ────────────────────────────────────────
# Um DataFrame é um conjunto de Series com o mesmo índice
dados_cidades = {
    'Cidade':     ['São Paulo', 'Fortaleza', 'Manaus', 'Salvador'],
    'Estado':     ['SP', 'CE', 'AM', 'BA'],
    'Populacao':  [12_325_232, 2_703_391, 1_440_000, 3_849_322],
    'IDHM':       [0.805, 0.754, 0.737, 0.759],
    'Capital':    [True, True, True, True]
}

df_exemplo = pd.DataFrame(dados_cidades)
print(df_exemplo)
print(f"\nFormato: {df_exemplo.shape}  →  ({df_exemplo.shape[0]} linhas × {df_exemplo.shape[1]} colunas)")


---
## 2.2 Teoria — Leitura e Inspeção do Dataset

Para carregar dados de um arquivo CSV usamos `pd.read_csv()`. Após o carregamento, os primeiros passos são sempre:
1. Verificar o tamanho (`shape`)
2. Ver as primeiras linhas (`head`)
3. Inspecionar tipos e nulos (`info`, `dtypes`, `isnull`)


In [ ]:
# ── Carregando o dataset Brazilian Cities ────────────────────────
# O arquivo CSV original tem um formato especial de aspas duplas.
# A função abaixo trata isso automaticamente.

import io

def carregar_brazilian_cities(caminho):
    """
    Lê o arquivo brazilian_city.csv que usa quoting especial
    (cada linha é envolta em aspas duplas com campos também aspas-duplos).
    Retorna um DataFrame limpo.
    """
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    
    # Separar linhas pelo delimitador CRLF
    linhas = raw.split('\r\n')
    
    # Remover a aspas externa de cada linha e desescapar aspas internas
    def corrigir_linha(linha):
        if linha.startswith('"') and linha.endswith('"'):
            linha = linha[1:-1]
        return linha.replace('""', '"')
    
    linhas_corrigidas = [corrigir_linha(l) for l in linhas if l.strip()]
    conteudo = '\n'.join(linhas_corrigidas)
    
    return pd.read_csv(io.StringIO(conteudo))

# Carrega o dataset
df = carregar_brazilian_cities('../datasets/brazilian_city.csv')

print(f"✅ Dataset carregado com sucesso!")
print(f"   Municípios: {df.shape[0]:,}")
print(f"   Variáveis:  {df.shape[1]}")


In [ ]:
# ── Primeiras linhas do dataset ─────────────────────────────────
# head(n) mostra as n primeiras linhas (padrão n=5)
df.head()


In [ ]:
# ── Últimas linhas ──────────────────────────────────────────────
df.tail(3)


In [ ]:
# ── Amostra aleatória ───────────────────────────────────────────
# Útil para ver uma visão representativa do dataset
df.sample(5, random_state=42)


In [ ]:
# ── Informações gerais do dataset ───────────────────────────────
# info() mostra: número de linhas, colunas, tipos de dados e uso de memória
df.info()


In [ ]:
# ── Tipos de dados de cada coluna ───────────────────────────────
print("Tipos de dados:")
print(df.dtypes.value_counts())
print()

# Colunas do tipo texto (object)
cols_texto = df.select_dtypes(include='object').columns.tolist()
print(f"Colunas de texto ({len(cols_texto)}): {cols_texto}")


In [ ]:
# ── Estatísticas descritivas ────────────────────────────────────
# describe() calcula: count, mean, std, min, quartis, max para numéricas
df.describe().round(2)


In [ ]:
# ── Dicionário de dados: entendendo as colunas principais ───────
dicionario = {
    'CITY':                 'Nome do município',
    'STATE':                'Estado (UF)',
    'CAPITAL':              '1 = Capital estadual, 0 = Não capital',
    'IBGE_RES_POP':         'População residente (Censo IBGE)',
    'ESTIMATED_POP':        'População estimada (mais recente)',
    'IDHM':                 'Índice de Desenvolvimento Humano Municipal',
    'IDHM_Renda':           'IDH - componente Renda',
    'IDHM_Longevidade':     'IDH - componente Longevidade',
    'IDHM_Educacao':        'IDH - componente Educação',
    'GDP':                  'PIB municipal (R$ mil)',
    'GDP_CAPITA':           'PIB per capita (R$)',
    'GVA_AGROPEC':          'Valor Adicionado Bruto — Agropecuária',
    'GVA_INDUSTRY':         'Valor Adicionado Bruto — Indústria',
    'GVA_SERVICES':         'Valor Adicionado Bruto — Serviços',
    'AREA':                 'Área territorial (km²)',
    'LONG':                 'Longitude geográfica',
    'LAT':                  'Latitude geográfica',
    'ALT':                  'Altitude (metros)',
    'Cars':                 'Número de automóveis registrados',
    'Motorcycles':          'Número de motocicletas registradas',
    'HOTELS':               'Número de hotéis',
    'BEDS':                 'Número de leitos hoteleiros',
    'RURAL_URBAN':          'Classificação: Urbano / Rural / Rural Adjacente',
}

print(f"{'COLUNA':<25} DESCRIÇÃO")
print("-" * 65)
for col, desc in dicionario.items():
    print(f"{col:<25} {desc}")


---
## 2.3 Teoria — Limpeza e Tratamento de Dados

Na prática, dados reais sempre têm problemas. As etapas clássicas de limpeza são:

| Problema | Solução Pandas |
|----------|---------------|
| Valores ausentes (NaN) | `fillna()`, `dropna()` |
| Duplicatas | `drop_duplicates()` |
| Tipos errados | `astype()`, `pd.to_numeric()` |
| Nomes inconsistentes | `.str.strip()`, `.str.upper()` |
| Outliers extremos | filtros booleanos, `clip()` |


In [ ]:
# ── 2.3.1 Verificando valores ausentes ──────────────────────────
nulos = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100).round(2)

# Montar relatório de nulos
relatorio_nulos = pd.DataFrame({
    'Ausentes': nulos,
    'Percentual (%)': nulos_pct
})

# Mostrar apenas colunas com pelo menos 1 nulo
relatorio_nulos = relatorio_nulos[relatorio_nulos['Ausentes'] > 0].sort_values('Ausentes', ascending=False)

if relatorio_nulos.empty:
    print("✅ Nenhum valor ausente encontrado no dataset!")
else:
    print(f"⚠️  {len(relatorio_nulos)} coluna(s) com valores ausentes:")
    print(relatorio_nulos)


In [ ]:
# ── 2.3.2 Verificando duplicatas ────────────────────────────────
n_duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {n_duplicatas}")

# Verificar duplicatas apenas pelo nome da cidade e estado
n_dup_cidade = df.duplicated(subset=['CITY', 'STATE']).sum()
print(f"Cidades com nome duplicado: {n_dup_cidade}")

# Ver quais cidades aparecem mais de uma vez
if n_dup_cidade > 0:
    cidades_dup = df[df.duplicated(subset=['CITY', 'STATE'], keep=False)].sort_values('CITY')
    print("\nExemplos:")
    print(cidades_dup[['CITY', 'STATE']].head(10))


In [ ]:
# ── 2.3.3 Criando uma cópia limpa do DataFrame ──────────────────
# Sempre trabalhe com uma cópia para preservar o original!
df_limpo = df.copy()

# Renomear colunas problemáticas para facilitar o uso
df_limpo = df_limpo.rename(columns={
    'IDHM Ranking 2010':    'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART':              'WALMART',
    'IBGE_1-4':              'IBGE_1a4',
    'IBGE_5-9':              'IBGE_5a9',
    'IBGE_10-14':            'IBGE_10a14',
    'IBGE_15-59':            'IBGE_15a59',
    'IBGE_60+':              'IBGE_60mais',
})

print("✅ Colunas renomeadas!")
print(f"Dataset limpo: {df_limpo.shape}")


In [ ]:
# ── 2.3.4 Verificando e corrigindo tipos de dados ───────────────
# Conferir colunas que deveriam ser numéricas
cols_numericas_esperadas = ['IDHM', 'GDP_CAPITA', 'AREA', 'ALT']

for col in cols_numericas_esperadas:
    print(f"{col:<20} → tipo atual: {df_limpo[col].dtype}")


In [ ]:
# ── 2.3.5 Tratando municípios com população zero ─────────────────
# Municípios com IBGE_RES_POP = 0 são dados inválidos
pop_zero = df_limpo[df_limpo['IBGE_RES_POP'] == 0]
print(f"Municípios com população IBGE = 0: {len(pop_zero)}")
if not pop_zero.empty:
    print(pop_zero[['CITY', 'STATE', 'IBGE_RES_POP', 'ESTIMATED_POP']].head())

# Usar população estimada onde a do IBGE é zero
df_limpo['POP_FINAL'] = df_limpo['IBGE_RES_POP'].where(
    df_limpo['IBGE_RES_POP'] > 0,   # condição: pop IBGE > 0 → usa IBGE
    other=df_limpo['ESTIMATED_POP']  # senão → usa estimada
)
print(f"\n✅ Coluna POP_FINAL criada com {(df_limpo['POP_FINAL'] > 0).sum()} municípios válidos")


In [ ]:
# ── 2.3.6 Criando coluna de Densidade Demográfica ───────────────
# Densidade = População / Área  (hab/km²)
# Evitar divisão por zero com mask
df_limpo['DENSIDADE'] = np.where(
    df_limpo['AREA'] > 0,
    (df_limpo['POP_FINAL'] / df_limpo['AREA']).round(2),
    np.nan
)

print("Densidade demográfica (hab/km²) — Top 10:")
print(df_limpo[['CITY', 'STATE', 'POP_FINAL', 'AREA', 'DENSIDADE']]
      .sort_values('DENSIDADE', ascending=False)
      .head(10)
      .to_string(index=False))


---
## 2.4 Teoria — Manipulação Básica e Estatística Descritiva

As operações mais frequentes em Pandas:

| Operação | Método |
|----------|--------|
| Selecionar colunas | `df[['col1', 'col2']]` |
| Filtrar linhas | `df[df['col'] > valor]` |
| Ordenar | `df.sort_values('col')` |
| Agrupar | `df.groupby('col').agg(...)` |
| Criar colunas | `df['nova'] = ...` |
| Pivot table | `df.pivot_table(...)` |


In [ ]:
# ── 2.4.1 Seleção de colunas ────────────────────────────────────
# Selecionar apenas as colunas de interesse
colunas_chave = ['CITY', 'STATE', 'POP_FINAL', 'IDHM', 'GDP_CAPITA', 'AREA', 'DENSIDADE', 'RURAL_URBAN']
df_chave = df_limpo[colunas_chave].copy()

print(f"DataFrame reduzido: {df_chave.shape}")
df_chave.head()


In [ ]:
# ── 2.4.2 Filtragem com condições ───────────────────────────────
# Filtros simples
capitais = df_limpo[df_limpo['CAPITAL'] == 1]
print(f"Capitais estaduais: {len(capitais)}")

# Filtro composto — municípios com IDHM alto E grande população
grandes_e_ricos = df_limpo[
    (df_limpo['IDHM'] >= 0.750) &
    (df_limpo['POP_FINAL'] >= 100_000)
]
print(f"Municípios com IDHM ≥ 0.75 e pop ≥ 100 mil: {len(grandes_e_ricos)}")

# Filtro com isin — municípios da Região Norte
estados_norte = ['AM', 'PA', 'RR', 'RO', 'AC', 'AP', 'TO']
norte = df_limpo[df_limpo['STATE'].isin(estados_norte)]
print(f"Municípios da Região Norte: {len(norte)}")


In [ ]:
# ── 2.4.3 Ordenação ─────────────────────────────────────────────
# Top 10 municípios com maior PIB per capita
top_pib = (df_limpo[['CITY', 'STATE', 'GDP_CAPITA', 'POP_FINAL', 'IDHM']]
           .sort_values('GDP_CAPITA', ascending=False)
           .head(10)
           .reset_index(drop=True))

top_pib.index += 1  # ranking começa em 1
print("🏆 Top 10 — Maior PIB per Capita:")
print(top_pib.to_string())


In [ ]:
# ── 2.4.4 Criando categorias com cut e qcut ─────────────────────
# cut() — intervalos fixos definidos pelo analista
df_limpo['IDHM_FAIXA'] = pd.cut(
    df_limpo['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1.0],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
)

contagem_faixas = df_limpo['IDHM_FAIXA'].value_counts().sort_index()
print("Municípios por faixa de IDHM:")
for faixa, n in contagem_faixas.items():
    barra = '█' * (n // 80)
    print(f"  {str(faixa):<12} {n:>5} {barra}")


In [ ]:
# ── 2.4.5 Agrupamentos com groupby ──────────────────────────────
# Estatísticas por Estado — usando agg() para múltiplas métricas
resumo_estado = df_limpo.groupby('STATE').agg(
    N_Municipios  = ('CITY',        'count'),
    Pop_Total     = ('POP_FINAL',   'sum'),
    IDHM_Medio   = ('IDHM',        'mean'),
    PIB_Medio    = ('GDP_CAPITA',  'mean'),
    Maior_Cidade  = ('POP_FINAL',   'max'),
).round(3)

# Ordenar por IDHM médio (descrescente)
resumo_estado = resumo_estado.sort_values('IDHM_Medio', ascending=False)
print("Resumo por Estado (Top 10 IDHM Médio):")
print(resumo_estado.head(10).to_string())


In [ ]:
# ── 2.4.6 Pivot Table — análise cruzada ────────────────────────
# Cruzar Região (Urbano/Rural) × Faixa de IDHM
pivot = pd.pivot_table(
    df_limpo,
    values='CITY',
    index='RURAL_URBAN',
    columns='IDHM_FAIXA',
    aggfunc='count',
    fill_value=0
)

print("Tabela Pivô — Quantidade de Municípios por Zona × IDHM:")
print(pivot.to_string())


In [ ]:
# ── 2.4.7 Estatísticas descritivas por grupo ────────────────────
# Comparar IDH das capitais vs demais municípios
df_limpo['TIPO'] = df_limpo['CAPITAL'].map({1: 'Capital', 0: 'Interior'})

stats_tipo = df_limpo.groupby('TIPO')[['IDHM', 'GDP_CAPITA', 'POP_FINAL']].describe().round(3)
print("Estatísticas: Capitais vs Interior")
print(stats_tipo[['IDHM', 'GDP_CAPITA']].to_string())


In [ ]:
# ── 2.4.8 Correlação entre variáveis ────────────────────────────
# Quais variáveis se correlacionam com o IDHM?
variaveis_analise = ['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'DENSIDADE', 'AREA',
                      'PAY_TV', 'FIXED_PHONES', 'Cars', 'Motorcycles']

corr = df_limpo[variaveis_analise].corr()['IDHM'].drop('IDHM').sort_values(ascending=False)

print("Correlação com IDHM:")
for var, val in corr.items():
    barra_pos = '▶' * int(abs(val) * 20) if val > 0 else ''
    barra_neg = '◀' * int(abs(val) * 20) if val < 0 else ''
    sinal = '+' if val > 0 else '-'
    print(f"  {var:<20} {sinal}{abs(val):.3f}  {barra_neg}{barra_pos}")


---
## 2.5 Exercícios Práticos — Módulo 2

> Use o DataFrame `df_limpo` para responder às questões abaixo.  
> O **gabarito** está no arquivo `gabarito_exercicios.ipynb`.

---

### Exercício 1 — Filtragem básica
Liste todos os municípios do **Ceará (CE)** com IDHM **maior que 0.700**, ordenados do maior para o menor IDHM. Mostre as colunas: CITY, IDHM, GDP_CAPITA.

```python
# Seu código aqui
```

---

### Exercício 2 — Agrupamento
Calcule, por **Estado**, a **soma da população** e o **IDHM médio**. Quais são os 5 estados mais populosos?

```python
# Seu código aqui
```

---

### Exercício 3 — Criação de coluna
Crie uma coluna chamada `FROTA_TOTAL` somando `Cars` e `Motorcycles`. Depois, calcule a **frota média por habitante** em cada estado (frota_total / pop_final).

```python
# Seu código aqui
```

---

### Exercício 4 — Análise de capitais
Compare o **PIB per capita médio** das capitais estaduais com o das demais cidades. Qual a diferença percentual?

```python
# Seu código aqui
```

---

### Exercício 5 — Estatísticas descritivas
Calcule média, mediana, desvio padrão e quartis do IDHM para as cidades classificadas como **"Urbano"**, **"Rural Adjacente"** e **"Rural Remoto"** separadamente.

```python
# Seu código aqui
```


In [ ]:
# Espaço para suas respostas

# Exercício 1

# Exercício 2

# Exercício 3

# Exercício 4

# Exercício 5


---
## 📌 Resumo — Comandos Essenciais do Pandas

```python
# Leitura
df = pd.read_csv('arquivo.csv')

# Inspeção
df.shape          # (linhas, colunas)
df.head()         # primeiras 5 linhas
df.info()         # tipos + nulos
df.describe()     # estatísticas numéricas

# Seleção
df['coluna']                    # uma coluna (Series)
df[['col1', 'col2']]            # múltiplas colunas
df.iloc[0:5]                    # por posição
df.loc[df['col'] > valor]       # por condição

# Limpeza
df.isnull().sum()               # contar nulos
df.dropna()                     # remover nulos
df.fillna(0)                    # preencher nulos
df.drop_duplicates()            # remover duplicatas
df.rename(columns={'a': 'b'})   # renomear

# Transformação
df['nova'] = df['col'] * 2      # nova coluna
pd.cut(df['col'], bins=[...])   # categorizar
df.groupby('col').agg(...)      # agrupar
df.pivot_table(...)             # tabela pivô
df.sort_values('col')           # ordenar
```

---
*Próximo: **Módulo 3 — Visualização de Dados***
